In [1]:
!pip install -r ./requirements.txt


In [6]:
import re
import pandas as pd
import numpy as np
import nltk
import sys
from nltk.corpus import stopwords
import random
from __future__ import print_function
import nltk
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/rairakesh/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

In [7]:

class Preprocessor :
    def __init__(self):
        print("*** PREPROCESSING ***")

    # Hashtags
    hash_regex = re.compile(r"#(\w+)")

    def hash_repl(self, match):
        return '__HASH_' + match.group(1).upper()

    # Handels
    hndl_regex = re.compile(r"@(\w+)")

    def hndl_repl(self, match):
        return '__HNDL'  # _'+match.group(1).upper()

    # URLs
    url_regex = re.compile(r"(http|https|ftp)://[a-zA-Z0-9\./]+")

    # Spliting by word boundaries
    word_bound_regex = re.compile(r"\W+")

    # Repeating words like hurrrryyyyyy
    rpt_regex = re.compile(r"(.)\1{1,}", re.IGNORECASE);

    def rpt_repl(self, match):
        return match.group(1) + match.group(1)

    # Emoticons
    emoticons = \
        [('__EMOT_SMILEY', [':-)', ':)', '(:', '(-:', ]), \
         ('__EMOT_LAUGH', [':-D', ':D', 'X-D', 'XD', 'xD', ]), \
         ('__EMOT_LOVE', ['<3', ':\*', ]), \
         ('__EMOT_WINK', [';-)', ';)', ';-D', ';D', '(;', '(-;', ]), \
         ('__EMOT_FROWN', [':-(', ':(', '(:', '(-:', ]), \
         ('__EMOT_CRY', [':,(', ':\'(', ':"(', ':((']), \
         ]

    # Punctuations
    punctuations = \
        [  # ('',		['.', ] )	,\
            # ('',		[',', ] )	,\
            # ('',		['\'', '\"', ] )	,\
            ('__PUNC_EXCL', ['!', ]), \
            ('__PUNC_QUES', ['?', ]), \
            ('__PUNC_ELLP', ['...', ]), \
            # FIXME : MORE? http://en.wikipedia.org/wiki/Punctuation
        ]

    # Printing functions for info
    def print_config(self, cfg):
        for (x, arr) in cfg:
            print(x, '\t')
            for a in arr:
                print (a, '\t')
            print ('')

    def print_emoticons(self):
        self.print_config(self.emoticons)

    def print_punctuations(self):
        self.print_config(self.punctuations)

    # For emoticon regexes
    def escape_paren(self, arr):
        return [text.replace(')', '[)}\]]').replace('(', '[({\[]') for text in arr]

    def regex_union(self, arr):
        return '(' + '|'.join(arr) + ')'

    def get_emoticons_regex(self):
        emoticons_regex = [(repl, re.compile(self.regex_union(self.escape_paren(regx)))) \
                           for (repl, regx) in self.emoticons]
        return emoticons_regex



    # For punctuation replacement
    def punctuations_repl(self, match):
        text = match.group(0)
        repl = []
        for (key, parr) in self.punctuations:
            for punc in parr:
                if punc in text:
                    repl.append(key)
        if (len(repl) > 0):
            return ' ' + ' '.join(repl) + ' '
        else:
            return ' '

    def processHashtags( self, text, subject='', query=[]):
        return re.sub( self.hash_regex, self.hash_repl, text )

    def processHandles( self, text, subject='', query=[]):
        return re.sub( self.hndl_regex, self.hndl_repl, text )

    def processUrls( self, text, subject='', query=[]):
        return re.sub( self.url_regex, ' __URL ', text )

    def processEmoticons( self, text, subject='', query=[]):
        for (repl, regx) in self.get_emoticons_regex() :
            text = re.sub(regx, '  ' +repl +' ', text)
        return text

    def processPunctuations( self, text, subject='', query=[]):
        return re.sub( self.word_bound_regex , self.punctuations_repl, text )

    def processRepeatings( 	self, text, subject='', query=[]):
        return re.sub( self.rpt_regex, self.rpt_repl, text )

    def processQueryTerm( self, text, subject='', query=[]):
        query_regex = "|".join([ re.escape(q) for q in query])
        return re.sub( query_regex, '__QUER', text, flags=re.IGNORECASE )

    def countHandles(self, text):
        return len( re.findall( self.hndl_regex, text) )
    def countHashtags(self, text):
        return len( re.findall( self.hash_regex, text) )
    def countUrls(self, text):
        return len( re.findall( self.url_regex, text) )
    def countEmoticons(self, text):
        count = 0
        for (repl, regx) in self.get_emoticons_regex() :
            count += len( re.findall( regx, text) )
        return count

    # FIXME: preprocessing.preprocess()! wtf! will need to move.
    # FIXME: use process functions inside
    def processAll(self, text, subject='', query=[]):

        if(len(query ) >0):
            query_regex = "|".join([ re.escape(q) for q in query])
            text = re.sub( query_regex, '__QUER', text, flags=re.IGNORECASE )

        text = re.sub( self.hash_regex, self.hash_repl, text )
        text = re.sub( self.hndl_regex, self.hndl_repl, text )
        text = re.sub( self.url_regex, ' __URL ', text )

        for (repl, regx) in self.get_emoticons_regex() :
            text = re.sub(regx, '  ' +repl +' ', text)


        text = text.replace('\'' ,'')
        # FIXME: Jugad

        text = re.sub( self.word_bound_regex , self.punctuations_repl, text )
        text = re.sub( self.rpt_regex, self.rpt_repl, text )

        return text

In [8]:
class DataUtil :

    def getTrainingAndTestData(self, reviews, K, k):

        from functools import wraps
       
        procReviews = reviews
        stemmer = nltk.stem.PorterStemmer()

        all_reviews = []  # DATADICT: all_reviews =   [ (words, sentiment), ... ]
        for tuple in procReviews.itertuples():
            # print(tuple[1] + tuple[2] + "\n")
            words = [word if (word[0:2] == '__') else word.lower() \
                     for word in tuple[2].split() \
                     if len(word) >= 3]
            words = [stemmer.stem(w) for w in words]  # DATADICT: words = [ 'word1', 'word2', ... ]
            all_reviews.append((words, tuple[1]))

        train_reviews = [x for i, x in enumerate(all_reviews) if i % K != k]
        test_reviews = [x for i, x in enumerate(all_reviews) if i % K == k]


        def get_word_features(words):
            bag = {}
            stop_words = set(stopwords.words('english'))
            filtered_words = [w for w in words if not w in stop_words]
            words_uni = ['has(%s)' % ug for ug in filtered_words]
            for f in words_uni:
                bag[f] = 1

            # bag = collections.Counter(words_uni+words_bi+words_tri)
            return bag

        negtn_regex = re.compile(r"""(?:
            ^(?:never|no|nothing|nowhere|noone|none|not|
                havent|hasnt|hadnt|cant|couldnt|shouldnt|
                wont|wouldnt|dont|doesnt|didnt|isnt|arent|aint
            )$
        )
        |
        n't
        """, re.X)

        pos_regex = re.compile(r"""(?:
                    ^(?:excellent|wow|awesome|happy|cool|good|love|
                        wonderful|amazing|amaze|bliss|enjoy|fantastic|
                        beautiful|beauty|better|very good|fun|funny|arent|luck|lucky|
                        nice|super|great
                    )$
                )
                |
                n't
                """, re.X)

        def get_negation_features(words):
            INF = 0.0
            negtn = [bool(negtn_regex.search(w)) for w in words]

            left = [0.0] * len(words)
            prev = 0.0
            for i in range(0, len(words)):
                if (negtn[i]):
                    prev = 1.0
                left[i] = prev
                prev = max(0.0, prev - 0.1)

            right = [0.0] * len(words)
            prev = 0.0
            for i in reversed(range(0, len(words))):
                if (negtn[i]):
                    prev = 1.0
                right[i] = prev
                prev = max(0.0, prev - 0.1)

            return dict(zip(
                ['neg_l(' + w + ')' for w in words] + ['neg_r(' + w + ')' for w in words],
                left + right))

        def get_positive_features(words):

            bag={}
            for word in words:
                if bool(pos_regex.search(word)):
                    key = 'pos(' + word + ')'
                    bag[key] = 1
            return bag


        def counter(func):  # http://stackoverflow.com/questions/13512391/to-count-no-times-a-function-is-called
            @wraps(func)
            def tmp(*args, **kwargs):
                tmp.count += 1
                return func(*args, **kwargs)

            tmp.count = 0
            return tmp

        @counter  # http://stackoverflow.com/questions/13512391/to-count-no-times-a-function-is-called
        def extract_features(words):

            features = {}
            negation_features = get_negation_features(words)
            features.update(negation_features)
            postive_features = get_positive_features(words)
            features.update(postive_features)
            word_features = get_word_features(words)
            features.update(word_features)
            sys.stderr.write('\rfeatures extracted for ' + str(extract_features.count) + ' reviews')
            return features

        extract_features.count = 0;
        reviews_processed = 0
        # Apply NLTK's Lazy Map
        print("length of train reviews "+str(len(train_reviews)))
        v_train = nltk.classify.apply_features(extract_features, train_reviews)
        print("length of test reviews " + str(len(test_reviews)))
        v_test = nltk.classify.apply_features(extract_features, test_reviews)
        return (v_train, v_test)

In [9]:

print("reading csv")
cust_data = pd.read_csv("./data/Customer_Sentiment.csv",encoding='ISO-8859-1',
                        names=["customer_id", "gender", "age_group", "region", "product_category",
                               "purchase_channel","platform","customer_rating","review_text","sentiment",
                               "response_time_hours","issue_resolved","complaint_registered"])

print("size of the  dataset")
print(cust_data.head())
print(cust_data.shape)

reviews = cust_data['review_text']
print("pre-processing started")
preprocess = Preprocessor()
processed_reviews = reviews.apply(lambda x: preprocess.processAll(x))
reviews_df = pd.DataFrame({'sentiment': cust_data['sentiment'],
                   'review': processed_reviews})
reviews_df.to_csv("./data/preprocessed_reviews.csv" ,index=False)
fid = open("./data/preprocessed_reviews.csv", "r")
li = fid.readlines()
fid.close()
print("shuffling started")
random.shuffle(li)
fid1 = open("./data/preprocessed_reviews_shuffled.csv", "w")
fid1.writelines(li)
fid1.close()
print("shuffled")
print("Calling main function")
reviews_processed = pd.read_csv("./data/preprocessed_reviews_shuffled.csv",
                             encoding='ISO-8859-1')
print("size of the pre processed dataset")
print(reviews_processed.head())
print(reviews_processed.shape)

print("further processing...")

data_util = DataUtil()
train, test = data_util.getTrainingAndTestData(reviews_processed,5, 1)

print(len(train))
print(len(test))

print("sample tests")
print(train[0])



reading csv
size of the  dataset
   customer_id  gender  age_group   region  product_category  \
0  customer_id  gender  age_group   region  product_category   
1            1    male        60+    north        automobile   
2            2   other      46-60  central             books   
3            3  female      36-45     east            sports   
4            4  female      18-25  central         groceries   

   purchase_channel              platform  customer_rating  \
0  purchase_channel              platform  customer_rating   
1            online              flipkart                1   
2            online      swiggy instamart                5   
3            online  facebook marketplace                1   
4            online                 zepto                2   

                               review_text  sentiment  response_time_hours  \
0                              review_text  sentiment  response_time_hours   
1      very disappointed with the quality.   negative

features extracted for 1 reviews